# Air quality preprocessing

Split **London** and **Los Angeles** from the global hourly file (years **2022–2025**), write city CSVs, then profile each output. One code cell per action.

## Setup

In [2]:
from pathlib import Path

import pandas as pd

In [3]:
def resolve_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "globaldata" / "Global_City_Air_Quality_Hourly.csv").exists():
        return cwd
    if (cwd / "data" / "globaldata" / "Global_City_Air_Quality_Hourly.csv").exists():
        return cwd / "data"
    raise FileNotFoundError(
        "Could not find Global_City_Air_Quality_Hourly.csv. "
        "Run the notebook with cwd = thesis_2026 or thesis_2026/data."
    )


DATA_DIR = resolve_data_dir()
GLOBAL_CSV = DATA_DIR / "globaldata" / "Global_City_Air_Quality_Hourly.csv"
OUT_LONDON = DATA_DIR / "london22_25.csv"
OUT_LA = DATA_DIR / "los_angeles22_25.csv"
YEARS = (2022, 2023, 2024, 2025)

## Load global dataset

In [4]:
df_all = pd.read_csv(GLOBAL_CSV)

In [5]:
df_all["time"] = pd.to_datetime(df_all["time"], utc=True, errors="coerce")

In [6]:
df_all["year"] = df_all["time"].dt.year

In [7]:
mask_years = df_all["year"].isin(YEARS)

## Build and save city files (2022–2025)

In [8]:
df_london = df_all.loc[mask_years & (df_all["city"] == "London")].copy()

In [9]:
df_london.drop(columns=["year"], inplace=True)

In [10]:
df_london.to_csv(OUT_LONDON, index=False)

In [11]:
df_la = df_all.loc[mask_years & (df_all["city"] == "Los Angeles")].copy()

In [12]:
df_la.drop(columns=["year"], inplace=True)

In [13]:
df_la.to_csv(OUT_LA, index=False)

---
## London `london22_25.csv` — load from disk

In [14]:
lon = pd.read_csv(OUT_LONDON, parse_dates=["time"])

In [15]:
lon["time"] = pd.to_datetime(lon["time"], utc=True, errors="coerce")

In [16]:
lon["year"] = lon["time"].dt.year

In [17]:
print("Number of rows:", len(lon))

Number of rows: 30048


In [18]:
print("Number of columns:", lon.shape[1])

Number of columns: 11


In [19]:
print("Column names:")
print(list(lon.columns))

Column names:
['time', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'city', 'latitude', 'longitude', 'year']


In [20]:
lon.dtypes

time                datetime64[us, UTC]
pm10                            float64
pm2_5                           float64
carbon_monoxide                 float64
nitrogen_dioxide                float64
sulphur_dioxide                 float64
ozone                           float64
city                                str
latitude                        float64
longitude                       float64
year                              int32
dtype: object

In [21]:
lon.info()

<class 'pandas.DataFrame'>
RangeIndex: 30048 entries, 0 to 30047
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   time              30048 non-null  datetime64[us, UTC]
 1   pm10              30048 non-null  float64            
 2   pm2_5             30048 non-null  float64            
 3   carbon_monoxide   30048 non-null  float64            
 4   nitrogen_dioxide  30048 non-null  float64            
 5   sulphur_dioxide   30048 non-null  float64            
 6   ozone             30048 non-null  float64            
 7   city              30048 non-null  str                
 8   latitude          30048 non-null  float64            
 9   longitude         30048 non-null  float64            
 10  year              30048 non-null  int32              
dtypes: datetime64[us, UTC](1), float64(8), int32(1), str(1)
memory usage: 2.4 MB


In [22]:
lon.head()

,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,city,latitude,longitude,year
0,2022-07-29 00:00:00+00:00,8.6,5.5,122.0,17.1,2.1,58.0,London,51.51,-0.13,2022
1,2022-07-29 01:00:00+00:00,9.4,6.5,137.0,15.0,1.7,57.0,London,51.51,-0.13,2022
2,2022-07-29 02:00:00+00:00,9.9,6.0,135.0,13.9,1.8,57.0,London,51.51,-0.13,2022
3,2022-07-29 03:00:00+00:00,10.0,6.2,132.0,13.2,1.8,53.0,London,51.51,-0.13,2022
4,2022-07-29 04:00:00+00:00,8.0,5.4,137.0,13.9,1.9,47.0,London,51.51,-0.13,2022


In [23]:
lon.tail()

,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,city,latitude,longitude,year
30043,2025-12-31 19:00:00+00:00,20.5,14.5,262.0,60.8,4.4,6.0,London,51.51,-0.13,2025
30044,2025-12-31 20:00:00+00:00,20.9,15.0,239.0,57.9,4.2,6.0,London,51.51,-0.13,2025
30045,2025-12-31 21:00:00+00:00,20.7,15.7,238.0,55.2,4.2,6.0,London,51.51,-0.13,2025
30046,2025-12-31 22:00:00+00:00,20.9,16.0,241.0,55.5,4.3,7.0,London,51.51,-0.13,2025
30047,2025-12-31 23:00:00+00:00,21.6,16.8,253.0,52.2,4.2,11.0,London,51.51,-0.13,2025


In [24]:
missing_counts = lon.isna().sum()
missing_counts

time                0
pm10                0
pm2_5               0
carbon_monoxide     0
nitrogen_dioxide    0
sulphur_dioxide     0
ozone               0
city                0
latitude            0
longitude           0
year                0
dtype: int64

In [25]:
missing_pct = (lon.isna().mean() * 100).round(2)
missing_pct

time                0.0
pm10                0.0
pm2_5               0.0
carbon_monoxide     0.0
nitrogen_dioxide    0.0
sulphur_dioxide     0.0
ozone               0.0
city                0.0
latitude            0.0
longitude           0.0
year                0.0
dtype: float64

In [26]:
print("Rows per year:")
print(lon["year"].value_counts().sort_index())

Rows per year:
year
2022    3744
2023    8760
2024    8784
2025    8760
Name: count, dtype: int64


In [27]:
print("Time range (min, max):", lon["time"].min(), lon["time"].max())

Time range (min, max): 2022-07-29 00:00:00+00:00 2025-12-31 23:00:00+00:00


In [28]:
dup_times = lon["time"].duplicated().sum()
print("Duplicate timestamps:", int(dup_times))

Duplicate timestamps: 0


In [29]:
numeric_cols = lon.select_dtypes(include="number").columns.tolist()
lon[numeric_cols].describe()

,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,latitude,longitude,year
count,30048.000000,30048.000000,30048.000000,30048.000000,30048.000000,30048.000000,30048.00,3.004800e+04,30048.000000
mean,13.593680,9.357894,186.610164,21.134708,2.920540,48.133390,51.51,-1.300000e-01,2023.750799
std,7.763587,6.491275,67.208576,14.959831,2.536289,23.744464,0.00,5.551207e-17,1.009655
min,1.200000,0.800000,59.000000,1.700000,0.300000,0.000000,51.51,-1.300000e-01,2022.000000
25%,8.500000,5.200000,146.000000,10.300000,1.400000,33.000000,51.51,-1.300000e-01,2023.000000
50%,11.400000,7.100000,170.000000,16.300000,2.200000,50.000000,51.51,-1.300000e-01,2024.000000
75%,16.400000,11.300000,205.000000,27.525000,3.500000,63.000000,51.51,-1.300000e-01,2025.000000
max,77.200000,62.700000,1028.000000,113.500000,41.300000,155.000000,51.51,-1.300000e-01,2025.000000


In [30]:
lon.drop(columns=["year"]).nunique()

time                30048
pm10                  548
pm2_5                 462
carbon_monoxide       540
nitrogen_dioxide      830
sulphur_dioxide       235
ozone                 155
city                    1
latitude                1
longitude               1
dtype: int64

In [31]:
print("Memory usage (MB):", round(lon.memory_usage(deep=True).sum() / 1e6, 2))

Memory usage (MB): 3.94


### London — 2024 & 2025: are all columns non-empty?

We treat a column as **fully populated** if it has **no nulls** and (for text columns) **no empty or whitespace-only strings**. Run this block after the London `lon` dataframe (with `year`) is built above.

In [ ]:
def completeness_report(df: pd.DataFrame) -> pd.DataFrame:
    """Per column: row count, nulls, blank strings, and whether every row has a value."""
    rows = len(df)
    records = []
    for col in df.columns:
        s = df[col]
        n_null = int(s.isna().sum())
        n_blank = 0
        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            st = s.astype("string")
            n_blank = int(((st.notna()) & (st.str.strip() == "")).sum())
        records.append(
            {
                "column": col,
                "n_rows": rows,
                "n_null": n_null,
                "n_blank_str": n_blank,
                "all_non_empty": n_null == 0 and n_blank == 0,
            }
        )
    return pd.DataFrame(records)

In [ ]:
lon_2024 = lon.loc[lon["year"] == 2024].copy()

In [ ]:
rep_2024 = completeness_report(lon_2024)
rep_2024

In [ ]:
all_ok_2024 = bool(rep_2024["all_non_empty"].all())
print("2024: every column fully populated?", all_ok_2024)
if not all_ok_2024:
    print(rep_2024.loc[~rep_2024["all_non_empty"]])

In [ ]:
lon_2025 = lon.loc[lon["year"] == 2025].copy()

In [ ]:
rep_2025 = completeness_report(lon_2025)
rep_2025

In [ ]:
all_ok_2025 = bool(rep_2025["all_non_empty"].all())
print("2025: every column fully populated?", all_ok_2025)
if not all_ok_2025:
    print(rep_2025.loc[~rep_2025["all_non_empty"]])

---
## Los Angeles `los_angeles22_25.csv` — load from disk

In [32]:
la = pd.read_csv(OUT_LA, parse_dates=["time"])

In [33]:
la["time"] = pd.to_datetime(la["time"], utc=True, errors="coerce")

In [34]:
la["year"] = la["time"].dt.year

In [35]:
print("Number of rows:", len(la))

Number of rows: 30048


In [36]:
print("Number of columns:", la.shape[1])

Number of columns: 11


In [37]:
print("Column names:")
print(list(la.columns))

Column names:
['time', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'city', 'latitude', 'longitude', 'year']


In [38]:
la.dtypes

time                datetime64[us, UTC]
pm10                            float64
pm2_5                           float64
carbon_monoxide                 float64
nitrogen_dioxide                float64
sulphur_dioxide                 float64
ozone                           float64
city                                str
latitude                        float64
longitude                       float64
year                              int32
dtype: object

In [39]:
la.info()

<class 'pandas.DataFrame'>
RangeIndex: 30048 entries, 0 to 30047
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   time              30048 non-null  datetime64[us, UTC]
 1   pm10              29911 non-null  float64            
 2   pm2_5             29911 non-null  float64            
 3   carbon_monoxide   29911 non-null  float64            
 4   nitrogen_dioxide  29911 non-null  float64            
 5   sulphur_dioxide   29911 non-null  float64            
 6   ozone             29911 non-null  float64            
 7   city              30048 non-null  str                
 8   latitude          30048 non-null  float64            
 9   longitude         30048 non-null  float64            
 10  year              30048 non-null  int32              
dtypes: datetime64[us, UTC](1), float64(8), int32(1), str(1)
memory usage: 2.4 MB


In [40]:
la.head()

,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,city,latitude,longitude,year
0,2022-07-29 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,Los Angeles,34.05,-118.24,2022
1,2022-07-29 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,Los Angeles,34.05,-118.24,2022
2,2022-07-29 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,Los Angeles,34.05,-118.24,2022
3,2022-07-29 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,Los Angeles,34.05,-118.24,2022
4,2022-07-29 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,Los Angeles,34.05,-118.24,2022


In [41]:
la.tail()

,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,city,latitude,longitude,year
30043,2025-12-31 19:00:00+00:00,21.4,21.3,325.0,58.9,11.4,1.0,Los Angeles,34.05,-118.24,2025
30044,2025-12-31 20:00:00+00:00,21.0,20.9,242.0,61.2,10.8,0.0,Los Angeles,34.05,-118.24,2025
30045,2025-12-31 21:00:00+00:00,20.7,20.6,200.0,62.9,10.7,0.0,Los Angeles,34.05,-118.24,2025
30046,2025-12-31 22:00:00+00:00,19.7,19.6,180.0,63.8,10.7,0.0,Los Angeles,34.05,-118.24,2025
30047,2025-12-31 23:00:00+00:00,18.3,18.3,166.0,62.3,10.3,0.0,Los Angeles,34.05,-118.24,2025


In [42]:
la.isna().sum()

time                  0
pm10                137
pm2_5               137
carbon_monoxide     137
nitrogen_dioxide    137
sulphur_dioxide     137
ozone               137
city                  0
latitude              0
longitude             0
year                  0
dtype: int64

In [43]:
(la.isna().mean() * 100).round(2)

time                0.00
pm10                0.46
pm2_5               0.46
carbon_monoxide     0.46
nitrogen_dioxide    0.46
sulphur_dioxide     0.46
ozone               0.46
city                0.00
latitude            0.00
longitude           0.00
year                0.00
dtype: float64

In [44]:
print("Rows per year:")
print(la["year"].value_counts().sort_index())

Rows per year:
year
2022    3744
2023    8760
2024    8784
2025    8760
Name: count, dtype: int64


In [45]:
print("Time range (min, max):", la["time"].min(), la["time"].max())

Time range (min, max): 2022-07-29 00:00:00+00:00 2025-12-31 23:00:00+00:00


In [46]:
print("Duplicate timestamps:", int(la["time"].duplicated().sum()))

Duplicate timestamps: 0


In [47]:
la_num_cols = la.select_dtypes(include="number").columns.tolist()
la[la_num_cols].describe()

,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,latitude,longitude,year
count,29911.000000,29911.000000,29911.000000,29911.000000,29911.000000,29911.000000,3.004800e+04,30048.00,30048.000000
mean,30.445889,22.429946,389.243857,50.301120,11.290565,47.548226,3.405000e+01,-118.24,2023.750799
std,19.998924,15.228720,280.678534,33.426439,6.153832,52.995351,1.421109e-14,0.00,1.009655
min,0.600000,0.400000,32.000000,0.000000,0.200000,-6.000000,3.405000e+01,-118.24,2022.000000
25%,18.100000,12.800000,214.000000,24.600000,6.800000,4.000000,3.405000e+01,-118.24,2023.000000
50%,25.500000,18.700000,297.000000,45.800000,10.200000,36.000000,3.405000e+01,-118.24,2024.000000
75%,36.700000,27.800000,468.000000,67.800000,14.700000,73.000000,3.405000e+01,-118.24,2025.000000
max,335.300000,328.800000,4700.000000,320.600000,77.300000,591.000000,3.405000e+01,-118.24,2025.000000


In [48]:
la.drop(columns=["year"]).nunique()

time                30048
pm10                  889
pm2_5                 768
carbon_monoxide      1489
nitrogen_dioxide     1768
sulphur_dioxide       416
ozone                 420
city                    1
latitude                1
longitude               1
dtype: int64

In [49]:
print("Memory usage (MB):", round(la.memory_usage(deep=True).sum() / 1e6, 2))

Memory usage (MB): 4.09


### London 2024 & 2025 — column completeness (no nulls, no blank text)

Loads `london22_25.csv` from disk. A column is **OK** only if every row has a non-null value and string columns have no empty/whitespace-only values.

In [50]:
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "london22_25.csv").exists():
    DATA_DIR = cwd
elif (cwd / "data" / "london22_25.csv").exists():
    DATA_DIR = cwd / "data"
else:
    raise FileNotFoundError("Put london22_25.csv in cwd or in cwd/data/")

LONDON_CSV = DATA_DIR / "london22_25.csv"

In [51]:
lon = pd.read_csv(LONDON_CSV, parse_dates=["time"])
lon["time"] = pd.to_datetime(lon["time"], utc=True, errors="coerce")
lon["year"] = lon["time"].dt.year

In [52]:
def completeness_report(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for col in df.columns:
        s = df[col]
        n_null = int(s.isna().sum())
        n_blank = 0
        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            st = s.astype("string")
            n_blank = int(((st.notna()) & (st.str.strip() == "")).sum())
        records.append(
            {
                "column": col,
                "n_rows": len(df),
                "n_null": n_null,
                "n_blank_str": n_blank,
                "all_non_empty": n_null == 0 and n_blank == 0,
            }
        )
    return pd.DataFrame(records)

In [53]:
lon_2024 = lon.loc[lon["year"] == 2024].copy()
rep_2024 = completeness_report(lon_2024)
rep_2024

,column,n_rows,n_null,n_blank_str,all_non_empty
0,time,8784,0,0,True
1,pm10,8784,0,0,True
2,pm2_5,8784,0,0,True
3,carbon_monoxide,8784,0,0,True
4,nitrogen_dioxide,8784,0,0,True
5,sulphur_dioxide,8784,0,0,True
6,ozone,8784,0,0,True
7,city,8784,0,0,True
8,latitude,8784,0,0,True
9,longitude,8784,0,0,True


In [54]:
all_ok_2024 = bool(rep_2024["all_non_empty"].all())
print("2024: every column fully populated?", all_ok_2024)
if not all_ok_2024:
    print(rep_2024.loc[~rep_2024["all_non_empty"]])

2024: every column fully populated? True


In [55]:
lon_2025 = lon.loc[lon["year"] == 2025].copy()
rep_2025 = completeness_report(lon_2025)
rep_2025

,column,n_rows,n_null,n_blank_str,all_non_empty
0,time,8760,0,0,True
1,pm10,8760,0,0,True
2,pm2_5,8760,0,0,True
3,carbon_monoxide,8760,0,0,True
4,nitrogen_dioxide,8760,0,0,True
5,sulphur_dioxide,8760,0,0,True
6,ozone,8760,0,0,True
7,city,8760,0,0,True
8,latitude,8760,0,0,True
9,longitude,8760,0,0,True


In [56]:
all_ok_2025 = bool(rep_2025["all_non_empty"].all())
print("2025: every column fully populated?", all_ok_2025)
if not all_ok_2025:
    print(rep_2025.loc[~rep_2025["all_non_empty"]])

2025: every column fully populated? True


### Export London 2024 and 2025 from `london22_25.csv`

Writes `london_2024.csv` and `london_2025.csv` next to the source file and prints row counts.

In [58]:
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "london22_25.csv").exists():
    DATA_DIR = cwd
elif (cwd / "data" / "london22_25.csv").exists():
    DATA_DIR = cwd / "data"
else:
    raise FileNotFoundError("Could not find london22_25.csv (expect cwd or cwd/data).")

SRC = DATA_DIR / "london22_25.csv"
OUT_2024 = DATA_DIR / "london_2024.csv"
OUT_2025 = DATA_DIR / "london_2025.csv"

In [59]:
df = pd.read_csv(SRC, parse_dates=["time"])
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df["year"] = df["time"].dt.year

In [60]:
df_2024 = df.loc[df["year"] == 2024].drop(columns=["year"])
len_2024 = len(df_2024)
print("london_2024.csv — number of rows:", len_2024)

london_2024.csv — number of rows: 8784


In [61]:
df_2024.to_csv(OUT_2024, index=False)
print("Saved:", OUT_2024.resolve())

Saved: /Users/new/Desktop/thesis_2026/data/london_2024.csv


In [62]:
df_2025 = df.loc[df["year"] == 2025].drop(columns=["year"])
len_2025 = len(df_2025)
print("london_2025.csv — number of rows:", len_2025)

london_2025.csv — number of rows: 8760


In [63]:
df_2025.to_csv(OUT_2025, index=False)
print("Saved:", OUT_2025.resolve())

Saved: /Users/new/Desktop/thesis_2026/data/london_2025.csv


In [64]:
print("Summary")
print("  london_2024.csv rows:", len_2024)
print("  london_2025.csv rows:", len_2025)

Summary
  london_2024.csv rows: 8784
  london_2025.csv rows: 8760
